# Non-IID Brain Tumor MRI (Dirichlet α = 0.5, 3 clients, 5 rounds)

**Experiment A — GPIFL:** CE + grid-wise Perona–Malik PIDL, λ = 0.5, same grid/hparams as the main setup except λ.

**Experiment B — FedAvg baseline:** `regularizer_type=none` (CE only), same non-IID partition.

**Partitioning:** Dirichlet label skew (`partitioning=dirichlet`, `dirichlet_alpha=0.5`) via `FL_RUN_OVERRIDE` → `get_federated_data()`.

**Outputs:** per-variant `final_model.pth`, logs under `results_non_iid/…`, summary `non_iid_comparison.csv`, optional IID reference row from `results/brain_tumor_mri/3_clients/` if present.

| Step | Action |
|------|--------|
| 1 | Clone + install |
| 2 | Download Brain Tumor MRI (KaggleHub) |
| 3 | Run both FL experiments + `finalize_experiment()` |
| 4 | Build CSV + zip + Colab download |

## 1 — Clone repository and dependencies

In [ ]:
GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'

import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/medical_fl_pidl')

if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
else:
    print('Repo already cloned. Pulling latest...')
    os.system(f'git -C {PROJECT_ROOT} pull')

if not PROJECT_ROOT.exists():
    raise RuntimeError(f'Clone failed: {GITHUB_REPO!r}')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'Working dir: {Path.cwd()}')

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt
!python -c "import flwr, torch, kagglehub; print(f'flwr={flwr.__version__}  torch={torch.__version__}')"
print('Dependencies OK.')

## 2 — Download Brain Tumor MRI only

In [ ]:
import kagglehub
from data.kaggle_loader import find_image_root

bt_path = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
DATA_ROOT = find_image_root(bt_path)
print('ImageFolder root:', DATA_ROOT)

## 3 — Experiment configuration and run loop

In [ ]:
import gc, json, os, time

import torch
from flwr.simulation import run_simulation

from federated.server_app import app as _server_app
from federated.client_app import app as _client_app

NUM_CLIENTS        = 3
NUM_SERVER_ROUNDS  = 5
LOCAL_EPOCHS       = 2
BATCH_SIZE         = 32
LEARNING_RATE      = 0.001
IMAGE_SIZE         = 224
RANDOM_SEED        = 42

PARTITIONING       = 'dirichlet'
DIRICHLET_ALPHA    = 0.5

FEATURE_LAYER      = 'layer2'
USE_GRID_LOSS      = True
GRID_SIZE          = 4
K_PM               = 1.0

SECAGG_MAX_WEIGHT = 1048575

RESULTS_BASE = PROJECT_ROOT / 'results_non_iid' / 'brain_tumor_mri' / f'dirichlet_a{DIRICHLET_ALPHA}_{NUM_CLIENTS}c'
RESULTS_BASE.mkdir(parents=True, exist_ok=True)

# Two runs: (folder name, regularizer, lambda_pm)
RUNS = [
    ('gpifl_lambda05', 'perona_malik', 0.5),
    ('fedavg_no_pidl', 'none',           0.0),
]

IID_REF_SUMMARY = PROJECT_ROOT / 'results' / 'brain_tumor_mri' / '3_clients' / 'fl_summary.json'

print('Results base:', RESULTS_BASE)

In [ ]:
def run_one_variant(variant_dir_name: str, reg_type: str, lam: float, resume: bool = True) -> dict:
    from federated.server_app import finalize_experiment

    log_dir = RESULTS_BASE / variant_dir_name
    st = dict(variant=variant_dir_name, log_dir=str(log_dir), skipped=False, success=False)

    if resume and (log_dir / 'fl_summary.json').exists():
        print(f'  [SKIP] {variant_dir_name} — fl_summary.json exists')
        st['skipped'] = st['success'] = True
        return st

    log_dir.mkdir(parents=True, exist_ok=True)
    recon = max(1, NUM_CLIENTS - 1)

    run_cfg = {
        'dataset_name':                    'brain_tumor_mri',
        'data_root':                       str(DATA_ROOT),
        'num_classes':                     '0',
        'num_clients':                     str(NUM_CLIENTS),
        'min_fit_clients':                 str(NUM_CLIENTS),
        'num_server_rounds':               str(NUM_SERVER_ROUNDS),
        'local_epochs':                    str(LOCAL_EPOCHS),
        'batch_size':                      str(BATCH_SIZE),
        'learning_rate':                   str(LEARNING_RATE),
        'image_size':                      str(IMAGE_SIZE),
        'feature_layer':                   FEATURE_LAYER,
        'regularizer_type':                reg_type,
        'lambda_pm':                       str(lam),
        'use_grid_loss':                   str(USE_GRID_LOSS and reg_type != 'none').lower(),
        'grid_size':                       str(GRID_SIZE),
        'k':                               str(K_PM),
        'random_seed':                     str(RANDOM_SEED),
        'log_dir':                         str(log_dir),
        'partitioning':                    PARTITIONING,
        'dirichlet_alpha':                 str(DIRICHLET_ALPHA),
        'secagg_num_shares':               str(NUM_CLIENTS),
        'secagg_reconstruction_threshold': str(recon),
        'secagg_max_weight':               str(SECAGG_MAX_WEIGHT),
    }

    os.environ['FL_RUN_OVERRIDE'] = json.dumps(run_cfg)
    gpu_frac = 0.5 if torch.cuda.is_available() else 0.0
    backend_cfg = {'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}}

    print(f'  [RUN ] {variant_dir_name} -> {log_dir}')
    t0 = time.time()
    try:
        run_simulation(
            server_app=_server_app,
            client_app=_client_app,
            num_supernodes=NUM_CLIENTS,
            backend_config=backend_cfg,
        )
        print(f'  [OK  ] {time.time() - t0:.0f}s')
        st['success'] = True
        finalize_experiment()
    except Exception as exc:
        print(f'  [FAIL] {exc}')
        import traceback; traceback.print_exc()
    finally:
        os.environ.pop('FL_RUN_OVERRIDE', None)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return st


for name, rt, lm in RUNS:
    print('\n===', name, '===')
    run_one_variant(name, rt, lm, resume=True)

## 4 — Summary CSV (non-IID + optional IID reference)

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for name, _, _ in RUNS:
    p = RESULTS_BASE / name / 'fl_summary.json'
    if not p.exists():
        print('Missing:', p)
        continue
    s = json.loads(p.read_text(encoding='utf-8'))
    rows.append({
        'variant':          name,
        'alpha':            DIRICHLET_ALPHA,
        'best_accuracy':    s.get('best_accuracy'),
        'best_macro_f1':    s.get('best_macro_f1'),
        'final_ece':        s.get('final_ece'),
        'final_roc_auc':    s.get('final_roc_auc_macro'),
    })

if IID_REF_SUMMARY.exists():
    s0 = json.loads(IID_REF_SUMMARY.read_text(encoding='utf-8'))
    rows.append({
        'variant':          'iid_reference_notebook01_3c',
        'alpha':            None,
        'best_accuracy':    s0.get('best_accuracy'),
        'best_macro_f1':    s0.get('best_macro_f1'),
        'final_ece':        s0.get('final_ece'),
        'final_roc_auc':    s0.get('final_roc_auc_macro'),
    })
    print('Appended IID reference from', IID_REF_SUMMARY)
else:
    print('IID reference not found (run notebook 01 for brain_tumor / 3_clients or copy fl_summary.json).')

df = pd.DataFrame(rows)
csv_path = RESULTS_BASE / 'non_iid_comparison.csv'
df.to_csv(csv_path, index=False)
print('Wrote', csv_path)
display(df)

## 5 — Zip `results_non_iid` and download (Colab)

In [ ]:
import shutil

zip_stem = str(PROJECT_ROOT / 'results_non_iid_archive')
archive = shutil.make_archive(zip_stem, 'zip', PROJECT_ROOT, 'results_non_iid')
print('Created', archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Not on Colab — zip saved at:', archive)